# Video Game Sales Analysis & Genre/Platform Success Prediction

This notebook implements the case study using R in Google Colab. It analyzes historical video-game sales by genre, platform, publisher, release year, and geographic region, then compares two baseline classifiers—**Decision Tree** and **Logistic Regression**—for a **High** versus **Low** global-sales category. It also uses **LIME (Local Interpretable Model-agnostic Explanations)** to explain the logistic-regression model's individual predictions.

## Before running in Google Colab

1. Open this notebook in Google Colab and select an **R runtime/kernel**.
2. Run each code cell from top to bottom.
3. The data-loading cell automatically downloads the public Kaggle dataset using the direct Kaggle API URL below, so you normally do **not** need to download or upload a CSV yourself:

   `https://www.kaggle.com/api/v1/datasets/download/anandshaw2001/video-game-sales`

4. The dataset is downloaded to the notebook's output folder, extracted, and checked for the expected columns before analysis begins. If Kaggle is temporarily unavailable or the public download endpoint changes, the notebook prints a clear error and gives you the Kaggle page link as a fallback.

The notebook writes CSV summaries, charts, fitted models, LIME explanations, and a findings report to `video_game_sales_outputs/`.

**Method note:** the model defines the High/Low label using the training-set median of `Global_Sales`. `Global_Sales` and sales-derived regional fields are excluded from model predictors to reduce target leakage. Results describe this historical dataset and do not guarantee future game success.


## 1. Setup and required packages

Installs and loads the required R packages, sets a reproducible random seed, and prints the project banner.


In [ ]:
# =====================================================================

options(stringsAsFactors = FALSE, repos = c(CRAN = "https://cloud.r-project.org"))
set.seed(42)

cat("\n============================================================\n")
cat(" VIDEO GAME SALES ANALYSIS & SUCCESS-CATEGORY PREDICTION\n")
cat("============================================================\n\n")

# ---------------------------------------------------------------------
# 1. Install/load required packages (including lime for local explanations)
# ---------------------------------------------------------------------
required_packages <- c("ggplot2", "rpart", "rpart.plot", "lime")
to_install <- required_packages[
  !vapply(required_packages, requireNamespace, quietly = TRUE, FUN.VALUE = logical(1))
]
if (length(to_install) > 0) {
  message("Installing required packages: ", paste(to_install, collapse = ", "))
  install.packages(to_install, dependencies = NA)
}

for (pkg in required_packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    stop("Required package could not be loaded: ", pkg,
         ". Check the runtime's internet connection and rerun this script.")
  }
}

suppressPackageStartupMessages(library(ggplot2))
if (!requireNamespace("lime", quietly = TRUE)) stop("The lime package is required for LIME explanations.")


## 2. Create output folders

Creates the output directory and a subfolder for plots. All CSV summaries, model files, and PNG charts are written here.


In [ ]:
# 2. Create output folders
# ---------------------------------------------------------------------
output_dir <- file.path(getwd(), "video_game_sales_outputs")
plot_dir <- file.path(output_dir, "plots")
dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(plot_dir, recursive = TRUE, showWarnings = FALSE)


## 3. Load and inspect the dataset

Finds an uploaded CSV or ZIP, normalizes common column names, loads the file, validates required fields, and reports dimensions, missingness, and exact duplicate rows.


In [ ]:
# 3. Column-name normalization and dataset discovery
# ---------------------------------------------------------------------
# The supplied project description says to verify the exact CSV schema.
# This helper supports the classic vgsales names and common alternatives.

column_aliases <- list(
  Rank = c("rank", "salesrank", "overallrank"),
  Name = c("name", "gamename", "title", "videogame", "gametitle"),
  Platform = c("platform", "console", "system", "gamingplatform"),
  Year = c("year", "releaseyear", "yearofrelease", "yearreleased"),
  Genre = c("genre", "category", "gamegenre"),
  Publisher = c("publisher", "publishingcompany", "gamepublisher"),
  NA_Sales = c("nasales", "nasalesmillions", "northamericasales", "northamericasalesmillions"),
  EU_Sales = c("eusales", "eusalesmillions", "europesales", "europesalesmillions"),
  JP_Sales = c("jpsales", "jpsalesmillions", "japansales", "japansalesmillions"),
  Other_Sales = c("othersales", "othersalesmillions", "restofworldsales"),
  Global_Sales = c("globalsales", "globalsalesmillions", "worldwidesales", "totalsales", "globalunits")
)

canonicalize_names <- function(old_names) {
  keys <- tolower(gsub("[^[:alnum:]]", "", old_names))
  new_names <- old_names
  for (canonical in names(column_aliases)) {
    idx <- which(keys %in% column_aliases[[canonical]])
    if (length(idx) > 0) new_names[idx[1]] <- canonical
  }
  make.unique(new_names)
}

schema_is_valid <- function(path) {
  header <- tryCatch(
    names(read.csv(path, nrows = 0, check.names = FALSE,
                   stringsAsFactors = FALSE, na.strings = c("", "NA", "N/A", "NULL", "null"))),
    error = function(e) character(0)
  )
  if (length(header) == 0) return(FALSE)
  canonical <- canonicalize_names(header)
  required <- c("Platform", "Year", "Genre", "Publisher", "Global_Sales")
  all(required %in% canonical)
}

# Download the public dataset directly from Kaggle if no compatible local
# CSV or uploaded ZIP is available. Kaggle announced public dataset downloads
# through its API without requiring authentication.
auto_download_kaggle_dataset <- function() {
  kaggle_url <- "https://www.kaggle.com/api/v1/datasets/download/anandshaw2001/video-game-sales"
  download_dir <- file.path(output_dir, "kaggle_download")
  zip_path <- file.path(download_dir, "video-game-sales.zip")
  extract_dir <- file.path(download_dir, "extracted")

  dir.create(download_dir, recursive = TRUE, showWarnings = FALSE)
  if (file.exists(zip_path)) unlink(zip_path)

  message("No compatible local CSV found. Downloading the public dataset from Kaggle...")
  message("Source: ", kaggle_url)

  zip_listing <- NULL
  download_methods <- unique(c(if (isTRUE(unname(capabilities("libcurl")))) "libcurl", "auto"))
  for (download_method in download_methods) {
    if (file.exists(zip_path)) unlink(zip_path)
    message("Attempting direct Kaggle download (method: ", download_method, ")...")

    download_ok <- tryCatch({
      utils::download.file(
        url = kaggle_url,
        destfile = zip_path,
        mode = "wb",
        method = download_method,
        quiet = FALSE
      )
      isTRUE(file.exists(zip_path) && !is.na(file.info(zip_path)$size) &&
               file.info(zip_path)$size > 1000)
    }, error = function(e) {
      message("Download attempt failed: ", conditionMessage(e))
      FALSE
    })

    if (download_ok) {
      zip_listing <- tryCatch(
        utils::unzip(zip_path, list = TRUE),
        error = function(e) NULL
      )
      if (!is.null(zip_listing) && nrow(zip_listing) > 0) break
      message("The returned file was not a valid ZIP archive; trying the next download method if available.")
    }
  }

  if (is.null(zip_listing) || nrow(zip_listing) == 0) {
    message("Automatic Kaggle download did not return a valid ZIP archive.")
    return(NULL)
  }

  if (dir.exists(extract_dir)) unlink(extract_dir, recursive = TRUE, force = TRUE)
  dir.create(extract_dir, recursive = TRUE, showWarnings = FALSE)

  extracted_ok <- tryCatch({
    utils::unzip(zip_path, exdir = extract_dir)
    TRUE
  }, error = function(e) {
    message("Could not extract the Kaggle ZIP: ", conditionMessage(e))
    FALSE
  })
  if (!extracted_ok) return(NULL)

  extracted_csvs <- list.files(
    extract_dir,
    pattern = "\\.csv$",
    full.names = TRUE,
    recursive = TRUE,
    ignore.case = TRUE
  )
  if (length(extracted_csvs) == 0) {
    message("The Kaggle archive did not contain a CSV file.")
    return(NULL)
  }

  valid_extracted <- extracted_csvs[vapply(extracted_csvs, schema_is_valid, logical(1))]
  if (length(valid_extracted) == 0) {
    message("CSV files were extracted, but none contained the required sales columns: Platform, Year, Genre, Publisher and Global_Sales.")
    message("CSV files found: ", paste(basename(extracted_csvs), collapse = ", "))
    return(NULL)
  }

  message("Kaggle dataset downloaded and validated: ", basename(valid_extracted[1]))
  valid_extracted[1]
}

find_dataset_csv <- function() {
  roots <- unique(c(getwd(), if (dir.exists("/content")) "/content"))
  csv_candidates <- character(0)
  zip_candidates <- character(0)
  for (root in roots) {
    csv_candidates <- c(csv_candidates, list.files(root, pattern = "\\.csv$",
                                                    full.names = TRUE, recursive = FALSE,
                                                    ignore.case = TRUE))
    zip_candidates <- c(zip_candidates, list.files(root, pattern = "\\.zip$",
                                                    full.names = TRUE, recursive = FALSE,
                                                    ignore.case = TRUE))
  }
  csv_candidates <- unique(csv_candidates[file.exists(csv_candidates)])
  preferred <- grepl("(^|[/\\\\])vgsales\\.csv$|video.?game.*sales.*\\.csv$|sales.*video.?game.*\\.csv$",
                     csv_candidates, ignore.case = TRUE)
  csv_candidates <- c(csv_candidates[preferred], csv_candidates[!preferred])
  valid <- csv_candidates[vapply(csv_candidates, schema_is_valid, logical(1))]
  if (length(valid) > 0) {
    message("Using compatible local dataset: ", basename(valid[1]))
    return(valid[1])
  }

  # If the user happened to upload a Kaggle ZIP, prefer that before network access.
  if (length(zip_candidates) > 0) {
    extract_dir <- file.path(output_dir, "input_extracted")
    dir.create(extract_dir, recursive = TRUE, showWarnings = FALSE)
    for (z in unique(zip_candidates)) {
      try(utils::unzip(z, exdir = extract_dir), silent = TRUE)
    }
    extracted_csvs <- list.files(extract_dir, pattern = "\\.csv$", full.names = TRUE,
                                 recursive = TRUE, ignore.case = TRUE)
    valid_extracted <- extracted_csvs[vapply(extracted_csvs, schema_is_valid, logical(1))]
    if (length(valid_extracted) > 0) {
      message("Using compatible dataset from uploaded ZIP: ", basename(valid_extracted[1]))
      return(valid_extracted[1])
    }
  }

  # Preferred flow: fetch the public Kaggle dataset automatically.
  downloaded_csv <- auto_download_kaggle_dataset()
  if (!is.null(downloaded_csv) && length(downloaded_csv) > 0 && file.exists(downloaded_csv)) {
    return(downloaded_csv)
  }

  stop(
    "Could not obtain a compatible video-game sales CSV automatically.\n",
    "The notebook tried the public Kaggle API URL:\n",
    "https://www.kaggle.com/api/v1/datasets/download/anandshaw2001/video-game-sales\n\n",
    "Please check that the Colab runtime has internet access. If Kaggle is unavailable, open\n",
    "https://www.kaggle.com/datasets/anandshaw2001/video-game-sales\n",
    "download the dataset ZIP, and upload it to the Colab Files panel as a temporary fallback.\n",
    "Required columns include Platform, Year, Genre, Publisher and Global_Sales."
  )
}

dataset_path <- find_dataset_csv()
cat("Dataset file selected: ", normalizePath(dataset_path, mustWork = FALSE), "\n", sep = "")

raw <- read.csv(
  dataset_path,
  stringsAsFactors = FALSE,
  check.names = FALSE,
  na.strings = c("", "NA", "N/A", "NULL", "null"),
  strip.white = TRUE
)
names(raw) <- canonicalize_names(names(raw))
rows_imported <- nrow(raw)
cols_imported <- ncol(raw)

required_columns <- c("Platform", "Year", "Genre", "Publisher", "Global_Sales")
missing_required <- setdiff(required_columns, names(raw))
if (length(missing_required) > 0) {
  stop("The selected CSV is missing required columns: ",
       paste(missing_required, collapse = ", "),
       ". Verify the file and its column names.")
}

if (!"Name" %in% names(raw)) {
  warning("No game-title column was found. Creating placeholder names for table output.")
  raw$Name <- paste0("Game_Record_", seq_len(nrow(raw)))
}

# Convert selected numeric columns carefully; nonnumeric values become NA.
numeric_columns <- intersect(
  c("Rank", "Year", "NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales", "Global_Sales"),
  names(raw)
)
for (col in numeric_columns) {
  vals <- trimws(as.character(raw[[col]]))
  vals[vals == ""] <- NA_character_
  vals <- gsub(",", "", vals, fixed = TRUE)
  raw[[col]] <- suppressWarnings(as.numeric(vals))
}

missing_by_column <- data.frame(
  Column = names(raw),
  Missing_Values = vapply(raw, function(x) as.integer(sum(is.na(x))), integer(1)),
  Percent_Missing = round(100 * vapply(raw, function(x) mean(is.na(x)), numeric(1)), 2),
  stringsAsFactors = FALSE
)
write.csv(missing_by_column, file.path(output_dir, "missing_values_by_column.csv"), row.names = FALSE)

rows_with_any_missing <- sum(!complete.cases(raw))
missing_cells_detected <- sum(is.na(raw))
exact_duplicate_rows <- sum(duplicated(raw))

cat("\n--- Initial data inspection ---\n")
cat("Rows imported: ", rows_imported, "\n", sep = "")
cat("Columns imported: ", cols_imported, "\n", sep = "")
cat("Missing cells after type conversion: ", missing_cells_detected, "\n", sep = "")
cat("Rows containing at least one missing value: ", rows_with_any_missing, "\n", sep = "")
cat("Exact duplicate rows detected: ", exact_duplicate_rows, "\n", sep = "")
cat("Column names after normalization: ", paste(names(raw), collapse = ", "), "\n", sep = "")


## 4. Data cleaning

Removes exact duplicate rows, excludes rows with missing or invalid global sales for the prediction task, and handles missing categorical labels.


In [ ]:
# 4. Data cleaning
# ---------------------------------------------------------------------
# Remove only exact duplicate rows. Repeated game names on different
# platforms/years are not automatically removed because they can be valid.
clean <- raw[!duplicated(raw), , drop = FALSE]
rows_removed_duplicates <- rows_imported - nrow(clean)

# The supervised target cannot be formed where Global_Sales is missing,
# infinite or negative; those rows are excluded and the count is reported.
invalid_global <- is.na(clean$Global_Sales) | !is.finite(clean$Global_Sales) | clean$Global_Sales < 0
rows_removed_invalid_global <- sum(invalid_global)
clean <- clean[!invalid_global, , drop = FALSE]
if (nrow(clean) < 20) {
  stop("Fewer than 20 usable rows remain after cleaning; verify the selected CSV.")
}

for (col in c("Genre", "Platform", "Publisher")) {
  clean[[col]] <- trimws(as.character(clean[[col]]))
  bad <- is.na(clean[[col]]) | !nzchar(clean[[col]])
  clean[[col]][bad] <- "Unknown"
}
clean$Name <- trimws(as.character(clean$Name))
clean$Name[is.na(clean$Name) | !nzchar(clean$Name)] <- "Unknown Title"

# Save a cleaned working copy. Year may remain NA for descriptive yearly
# analysis; it is imputed with the training-set median only for modelling.
write.csv(clean, file.path(output_dir, "cleaned_video_game_sales.csv"), row.names = FALSE)

quality_report <- data.frame(
  Metric = c(
    "Rows imported", "Columns imported", "Missing cells detected after type conversion",
    "Rows with at least one missing value", "Exact duplicate rows detected",
    "Rows removed as exact duplicates", "Rows removed for missing/invalid Global_Sales",
    "Rows retained after cleaning", "Missing Year values retained for non-year analyses"
  ),
  Value = c(
    rows_imported, cols_imported, missing_cells_detected, rows_with_any_missing,
    exact_duplicate_rows, rows_removed_duplicates, rows_removed_invalid_global,
    nrow(clean), sum(is.na(clean$Year))
  ),
  stringsAsFactors = FALSE
)
write.csv(quality_report, file.path(output_dir, "data_quality_summary.csv"), row.names = FALSE)

cat("\n--- Cleaning summary ---\n")
print(quality_report, row.names = FALSE)
cat("Categorical missing values in Genre/Platform/Publisher were represented as 'Unknown'.\n")
cat("Global_Sales is the sales target source; sales columns are not used as predictors.\n")


## 5. Exploratory data analysis and summary tables

Calculates genre, platform, publisher, yearly, regional, and top-game summaries from the selected CSV.


In [ ]:
# 5. Exploratory data analysis (EDA): group summaries
# ---------------------------------------------------------------------
make_group_summary <- function(data, group_col) {
  group_values <- as.character(data[[group_col]])
  group_values[is.na(group_values) | !nzchar(trimws(group_values))] <- "Unknown"
  split_sales <- split(data$Global_Sales, group_values)
  result <- do.call(rbind, lapply(names(split_sales), function(group_name) {
    x <- split_sales[[group_name]]
    data.frame(
      Category = group_name,
      Game_Count = length(x),
      Total_Global_Sales = sum(x, na.rm = TRUE),
      Mean_Global_Sales = mean(x, na.rm = TRUE),
      Median_Global_Sales = median(x, na.rm = TRUE),
      stringsAsFactors = FALSE
    )
  }))
  result <- result[order(result$Total_Global_Sales, decreasing = TRUE), , drop = FALSE]
  names(result)[1] <- group_col
  rownames(result) <- NULL
  result
}

genre_summary <- make_group_summary(clean, "Genre")
platform_summary <- make_group_summary(clean, "Platform")
publisher_summary <- make_group_summary(clean, "Publisher")
write.csv(genre_summary, file.path(output_dir, "genre_sales_summary.csv"), row.names = FALSE)
write.csv(platform_summary, file.path(output_dir, "platform_sales_summary.csv"), row.names = FALSE)
write.csv(publisher_summary, file.path(output_dir, "publisher_sales_summary.csv"), row.names = FALSE)

# Year summaries exclude records with unknown/non-numeric Year rather than
# assigning them an artificial release year.
valid_year_rows <- clean[!is.na(clean$Year) & is.finite(clean$Year), , drop = FALSE]
if (nrow(valid_year_rows) > 0) {
  year_groups <- split(valid_year_rows, as.character(valid_year_rows$Year))
  yearly_summary <- do.call(rbind, lapply(names(year_groups), function(year_label) {
    sub <- year_groups[[year_label]]
    data.frame(
      Year = as.numeric(year_label),
      Game_Count = nrow(sub),
      Total_Global_Sales = sum(sub$Global_Sales, na.rm = TRUE),
      Mean_Global_Sales = mean(sub$Global_Sales, na.rm = TRUE),
      stringsAsFactors = FALSE
    )
  }))
  yearly_summary <- yearly_summary[order(yearly_summary$Year), , drop = FALSE]
  rownames(yearly_summary) <- NULL
} else {
  yearly_summary <- data.frame(Year = numeric(0), Game_Count = integer(0),
                               Total_Global_Sales = numeric(0), Mean_Global_Sales = numeric(0))
}
write.csv(yearly_summary, file.path(output_dir, "yearly_sales_summary.csv"), row.names = FALSE)

regional_columns <- intersect(c("NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"), names(clean))
if (length(regional_columns) > 0) {
  regional_summary <- data.frame(
    Region = regional_columns,
    Total_Regional_Sales = vapply(regional_columns, function(col) sum(clean[[col]], na.rm = TRUE), numeric(1)),
    Records_With_Region_Value = vapply(regional_columns, function(col) as.integer(sum(!is.na(clean[[col]]))), integer(1)),
    stringsAsFactors = FALSE
  )
  regional_summary <- regional_summary[order(regional_summary$Total_Regional_Sales, decreasing = TRUE), , drop = FALSE]
} else {
  regional_summary <- data.frame(Region = character(0), Total_Regional_Sales = numeric(0),
                                 Records_With_Region_Value = integer(0))
  message("Regional sales columns were not found; regional comparison will be skipped.")
}
write.csv(regional_summary, file.path(output_dir, "regional_sales_summary.csv"), row.names = FALSE)

# Top individual recorded game/platform rows (not aggregated franchise totals).
top_games <- clean[order(clean$Global_Sales, decreasing = TRUE), , drop = FALSE]
select_top_cols <- intersect(c("Name", "Platform", "Year", "Genre", "Publisher", "NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales", "Global_Sales"), names(top_games))
top_games <- head(top_games[, select_top_cols, drop = FALSE], 10)
write.csv(top_games, file.path(output_dir, "top_10_games_by_global_sales.csv"), row.names = FALSE)

cat("\n--- Genre summary (top 10 by total Global_Sales) ---\n")
print(utils::head(genre_summary, 10), row.names = FALSE)
cat("\n--- Platform summary (top 10 by total Global_Sales) ---\n")
print(utils::head(platform_summary, 10), row.names = FALSE)
cat("\n--- Top 10 recorded game/platform entries ---\n")
print(top_games, row.names = FALSE)


## 6. Visual Exploratory Data Analysis (charts shown inline)

Creates charts for data quality, genre/platform/publisher sales, yearly release trends, regional sales, global-sales distribution, and top game records. Each chart is both saved as a PNG in `video_game_sales_outputs/plots/` and displayed directly below this code cell in Colab.


In [ ]:
# 6. Exploratory charts: save each chart and display it inline
# ---------------------------------------------------------------------
base_theme <- theme_minimal(base_size = 12) +
  theme(plot.title = element_text(face = "bold", size = 14),
        axis.title = element_text(face = "bold"),
        panel.grid.minor = element_blank())

save_plot <- function(plot_object, filename, width = 10, height = 6) {
  ggsave(filename = file.path(plot_dir, filename), plot = plot_object,
         width = width, height = height, units = "in", dpi = 160, bg = "white")
}

# Chart 1: missing values by source column
missing_df <- data.frame(
  Column = names(raw),
  Missing_Count = as.integer(colSums(is.na(raw))),
  stringsAsFactors = FALSE
)
missing_df <- missing_df[missing_df$Missing_Count > 0, , drop = FALSE]
if (nrow(missing_df) > 0) {
  missing_df <- missing_df[order(missing_df$Missing_Count), , drop = FALSE]
  p_missing <- ggplot(missing_df, aes(x = reorder(Column, Missing_Count), y = Missing_Count)) +
    geom_col(fill = "#E58A2B") + coord_flip() +
    labs(title = "Missing Values by Column (Before Cleaning)", x = NULL, y = "Missing cells") + base_theme
  save_plot(p_missing, "missing_values_by_column.png", width = 9, height = 6)
  print(p_missing)
} else {
  message("No missing cells were detected in the imported dataset.")
}

# Chart 2: genre sales volume
plot_genre <- head(genre_summary, 12)
p_genre <- ggplot(plot_genre, aes(x = reorder(Genre, Total_Global_Sales), y = Total_Global_Sales)) +
  geom_col(fill = "#3478C8") + coord_flip() +
  labs(title = "Global Sales by Genre", x = "Genre", y = "Total Global_Sales (source units)") + base_theme
save_plot(p_genre, "genre_global_sales.png")
print(p_genre)

# Chart 3: genre median sales (less sensitive to blockbuster outliers)
plot_genre_median <- head(genre_summary[order(genre_summary$Median_Global_Sales, decreasing = TRUE), , drop = FALSE], 12)
p_genre_median <- ggplot(plot_genre_median, aes(x = reorder(Genre, Median_Global_Sales), y = Median_Global_Sales)) +
  geom_col(fill = "#7B61A8") + coord_flip() +
  labs(title = "Median Global Sales by Genre", x = "Genre", y = "Median Global_Sales (source units)") + base_theme
save_plot(p_genre_median, "genre_median_global_sales.png")
print(p_genre_median)

# Chart 4: platform sales volume
plot_platform <- head(platform_summary, 12)
p_platform <- ggplot(plot_platform, aes(x = reorder(Platform, Total_Global_Sales), y = Total_Global_Sales)) +
  geom_col(fill = "#18A999") + coord_flip() +
  labs(title = "Top Platforms by Global Sales", x = "Platform", y = "Total Global_Sales (source units)") + base_theme
save_plot(p_platform, "platform_global_sales.png")
print(p_platform)

# Chart 5: publishers by total sales
plot_publisher <- head(publisher_summary, 12)
p_publisher <- ggplot(plot_publisher, aes(x = reorder(Publisher, Total_Global_Sales), y = Total_Global_Sales)) +
  geom_col(fill = "#E58A2B") + coord_flip() +
  labs(title = "Top Publishers by Global Sales", x = "Publisher", y = "Total Global_Sales (source units)") + base_theme
save_plot(p_publisher, "publisher_global_sales.png")
print(p_publisher)

# Charts 6 and 7: yearly sales and release counts
if (nrow(yearly_summary) > 0) {
  p_year <- ggplot(yearly_summary, aes(x = Year, y = Total_Global_Sales)) +
    geom_line(color = "#3478C8", linewidth = 0.9) +
    geom_point(color = "#18A999", size = 1.8) +
    labs(title = "Recorded Global Sales by Release Year", x = "Release year",
         y = "Total Global_Sales (source units)") + base_theme
  save_plot(p_year, "yearly_global_sales.png")
  print(p_year)

  p_count <- ggplot(yearly_summary, aes(x = Year, y = Game_Count)) +
    geom_line(color = "#E58A2B", linewidth = 0.9) +
    geom_point(color = "#7B61A8", size = 1.8) +
    labs(title = "Number of Game Records by Release Year", x = "Release year", y = "Record count") + base_theme
  save_plot(p_count, "yearly_release_counts.png")
  print(p_count)
} else {
  message("No valid release-year values were available; yearly charts were skipped.")
}

# Chart 8: regional comparison, when the source file contains regional fields
if (nrow(regional_summary) > 0) {
  p_region <- ggplot(regional_summary, aes(x = reorder(Region, Total_Regional_Sales), y = Total_Regional_Sales)) +
    geom_col(fill = "#D95F02") + coord_flip() +
    labs(title = "Regional Sales Comparison", x = "Region", y = "Total regional sales (source units)") + base_theme
  save_plot(p_region, "regional_global_sales.png")
  print(p_region)
} else {
  message("Regional sales chart skipped because no supported regional columns were found.")
}

# Chart 9: global-sales distribution. This is often right-skewed.
p_distribution <- ggplot(clean, aes(x = Global_Sales)) +
  geom_histogram(bins = 40, fill = "#3478C8", color = "white") +
  labs(title = "Distribution of Global Sales", x = "Global_Sales (source units)", y = "Number of records") + base_theme
save_plot(p_distribution, "global_sales_distribution.png")
print(p_distribution)

# Chart 10: log-scaled view makes the long tail easier to inspect.
p_distribution_log <- ggplot(clean, aes(x = log10(Global_Sales + 0.01))) +
  geom_histogram(bins = 40, fill = "#18A999", color = "white") +
  labs(title = "Global Sales Distribution (Log10 View)", x = "log10(Global_Sales + 0.01)", y = "Number of records") + base_theme
save_plot(p_distribution_log, "global_sales_distribution_log10.png")
print(p_distribution_log)

# Chart 11: top 10 individual game/platform records
p_top_games <- top_games
p_top_games$Display_Label <- paste0(p_top_games$Name, " (", p_top_games$Platform, ")")
p_top_games <- p_top_games[order(p_top_games$Global_Sales), , drop = FALSE]
p_top <- ggplot(p_top_games, aes(x = reorder(Display_Label, Global_Sales), y = Global_Sales)) +
  geom_col(fill = "#7B61A8") + coord_flip() +
  labs(title = "Top 10 Recorded Game/Platform Entries", x = "Game (platform)",
       y = "Global_Sales (source units)") + base_theme
save_plot(p_top, "top_10_games.png", width = 11, height = 7)
print(p_top)

cat("\nEDA charts saved to: ", normalizePath(plot_dir, mustWork = FALSE), "\n", sep = "")


## 7. Train and evaluate baseline sales classifiers

Defines High/Low success using the training-set median `Global_Sales` threshold, prepares predictor data, and fits a baseline Decision Tree. The following comparison section adds Logistic Regression so their held-out performance can be compared fairly. `Global_Sales`, rank, and regional-sales columns are excluded from the predictors to avoid direct target leakage.


In [ ]:
# 7. Build a high-sales / low-sales target and train a baseline tree model
# ---------------------------------------------------------------------
# The threshold is calculated from TRAINING data only. Global_Sales, Rank,
# and regional sales are never used as predictors, preventing direct target
# leakage. The allowed predictors are Genre, Platform, Publisher and Year.

set.seed(42)
if (nrow(clean) < 40) stop("At least 40 usable rows are recommended for the train/test model.")
train_indices <- sample(seq_len(nrow(clean)), size = floor(0.75 * nrow(clean)) )
train_raw <- clean[train_indices, , drop = FALSE]
test_raw <- clean[-train_indices, , drop = FALSE]

sales_threshold <- median(train_raw$Global_Sales, na.rm = TRUE)
train_class <- factor(ifelse(train_raw$Global_Sales >= sales_threshold, "High", "Low"),
                      levels = c("Low", "High"))
test_class <- factor(ifelse(test_raw$Global_Sales >= sales_threshold, "High", "Low"),
                     levels = c("Low", "High"))
if (length(unique(train_class)) < 2) {
  stop("The training target has only one sales class at the median threshold. ",
       "Inspect Global_Sales values or choose a documented alternative threshold.")
}

# Collapse rare categorical levels using TRAINING data only. Unseen test
# categories are mapped to Other, avoiding unseen-factor-level errors.
prepare_factor_pair <- function(train_values, test_values, min_count = 5) {
  tr <- trimws(as.character(train_values))
  te <- trimws(as.character(test_values))
  tr[is.na(tr) | !nzchar(tr)] <- "Unknown"
  te[is.na(te) | !nzchar(te)] <- "Unknown"
  counts <- table(tr)
  keep <- names(counts[counts >= min_count])
  if (length(keep) == 0) keep <- names(counts)[which.max(counts)]
  tr2 <- ifelse(tr %in% keep, tr, "Other")
  te2 <- ifelse(te %in% keep, te, "Other")
  levs <- unique(c(keep, "Other"))
  list(train = factor(tr2, levels = levs), test = factor(te2, levels = levs))
}

genre_pair <- prepare_factor_pair(train_raw$Genre, test_raw$Genre, min_count = 5)
platform_pair <- prepare_factor_pair(train_raw$Platform, test_raw$Platform, min_count = 5)
publisher_pair <- prepare_factor_pair(train_raw$Publisher, test_raw$Publisher, min_count = 10)

year_median <- median(train_raw$Year, na.rm = TRUE)
if (!is.finite(year_median)) year_median <- 0
train_year <- ifelse(is.na(train_raw$Year) | !is.finite(train_raw$Year), year_median, train_raw$Year)
test_year <- ifelse(is.na(test_raw$Year) | !is.finite(test_raw$Year), year_median, test_raw$Year)

train_model <- data.frame(
  Sales_Class = train_class,
  Genre = genre_pair$train,
  Platform = platform_pair$train,
  Publisher = publisher_pair$train,
  Year = as.numeric(train_year)
)
test_model <- data.frame(
  Sales_Class = test_class,
  Genre = genre_pair$test,
  Platform = platform_pair$test,
  Publisher = publisher_pair$test,
  Year = as.numeric(test_year)
)

cat("\n--- Model setup ---\n")
cat("Training rows: ", nrow(train_model), " | Test rows: ", nrow(test_model), "\n", sep = "")
cat("Training-set median Global_Sales threshold: ", signif(sales_threshold, 5), "\n", sep = "")
cat("Training class balance:\n")
print(round(prop.table(table(train_model$Sales_Class)), 3))
cat("Test class balance:\n")
print(round(prop.table(table(test_model$Sales_Class)), 3))
cat("Predictors: Genre, Platform, Publisher and Year. Global_Sales is excluded from model inputs.\n")

sales_tree <- rpart::rpart(
  Sales_Class ~ Genre + Platform + Publisher + Year,
  data = train_model,
  method = "class",
  control = rpart::rpart.control(cp = 0.005, minsplit = 20, maxdepth = 5, xval = 5)
)

predicted_class <- predict(sales_tree, newdata = test_model, type = "class")
predicted_probabilities <- predict(sales_tree, newdata = test_model, type = "prob")
if (!"High" %in% colnames(predicted_probabilities)) {
  stop("The fitted tree did not produce a High-class probability. Check target class balance.")
}
high_probability <- as.numeric(predicted_probabilities[, "High"])
predicted_class <- factor(as.character(predicted_class), levels = c("Low", "High"))

confusion <- table(
  Actual = factor(test_model$Sales_Class, levels = c("Low", "High")),
  Predicted = factor(predicted_class, levels = c("Low", "High"))
)
write.csv(as.data.frame.matrix(confusion), file.path(output_dir, "confusion_matrix.csv"), row.names = TRUE)
cat("\n--- Confusion matrix (test set) ---\n")
print(confusion)

safe_ratio <- function(numerator, denominator) {
  if (denominator == 0) NA_real_ else numerator / denominator
}

tn <- as.numeric(confusion["Low", "Low"])
fp <- as.numeric(confusion["Low", "High"])
fn <- as.numeric(confusion["High", "Low"])
tp <- as.numeric(confusion["High", "High"])
accuracy <- safe_ratio(tp + tn, sum(confusion))
precision_high <- safe_ratio(tp, tp + fp)
recall_high <- safe_ratio(tp, tp + fn)
specificity_low <- safe_ratio(tn, tn + fp)
f1_high <- if (is.na(precision_high) || is.na(recall_high) || precision_high + recall_high == 0) {
  NA_real_
} else {
  2 * precision_high * recall_high / (precision_high + recall_high)
}

auc_from_ranks <- function(prob, actual, positive_label = "High") {
  keep <- is.finite(prob) & !is.na(actual)
  prob <- prob[keep]
  actual <- as.character(actual[keep])
  positive <- actual == positive_label
  n_pos <- sum(positive)
  n_neg <- sum(!positive)
  if (n_pos == 0 || n_neg == 0) return(NA_real_)
  ranks <- rank(prob, ties.method = "average")
  as.numeric((sum(ranks[positive]) - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))
}
auc <- auc_from_ranks(high_probability, test_model$Sales_Class)

model_metrics <- data.frame(
  Model = "Decision Tree (rpart)",
  Train_Rows = nrow(train_model),
  Test_Rows = nrow(test_model),
  Training_Median_Global_Sales_Threshold = sales_threshold,
  Classification_Threshold = 0.50,
  Accuracy = accuracy,
  Precision_High = precision_high,
  Recall_High = recall_high,
  Specificity_Low = specificity_low,
  F1_High = f1_high,
  AUC_High = auc,
  stringsAsFactors = FALSE
)
write.csv(model_metrics, file.path(output_dir, "model_metrics.csv"), row.names = FALSE)

cat("\n--- Model performance (held-out test set) ---\n")
print(model_metrics, row.names = FALSE)


## 8. Logistic Regression and model comparison

Fits a second baseline classifier using the same train/test split and sales-success target. For this model, less-frequent platform and publisher levels are grouped more aggressively to make the categorical model more stable. The cell reports the Logistic Regression confusion matrix and compares Accuracy, Precision, Recall, Specificity, F1-score, and AUC against the Decision Tree.


In [ ]:
# 8. Logistic regression baseline and model comparison
# ---------------------------------------------------------------------
# Logistic regression is a second, interpretable baseline. The response
# factor levels are Low then High, so glm(binomial) models P(High).
# Group rarer platform/publisher categories more aggressively than the tree
# to reduce sparse one-hot columns in the logistic model.

platform_glm_pair <- prepare_factor_pair(train_raw$Platform, test_raw$Platform, min_count = 30)
publisher_glm_pair <- prepare_factor_pair(train_raw$Publisher, test_raw$Publisher, min_count = 100)
genre_glm_pair <- prepare_factor_pair(train_raw$Genre, test_raw$Genre, min_count = 5)

train_glm_model <- data.frame(
  Sales_Class = factor(train_class, levels = c("Low", "High")),
  Genre = genre_glm_pair$train,
  Platform = platform_glm_pair$train,
  Publisher = publisher_glm_pair$train,
  Year = as.numeric(train_model$Year)
)
test_glm_model <- data.frame(
  Sales_Class = factor(test_class, levels = c("Low", "High")),
  Genre = genre_glm_pair$test,
  Platform = platform_glm_pair$test,
  Publisher = publisher_glm_pair$test,
  Year = as.numeric(test_model$Year)
)

set.seed(42)
sales_logistic <- stats::glm(
  Sales_Class ~ Genre + Platform + Publisher + Year,
  data = train_glm_model,
  family = stats::binomial(link = "logit"),
  control = stats::glm.control(maxit = 100)
)

logistic_high_probability <- as.numeric(stats::predict(
  sales_logistic, newdata = test_glm_model, type = "response"
))
if (any(!is.finite(logistic_high_probability))) {
  warning("Some Logistic Regression probabilities are not finite; those rows are excluded from its metric calculations.")
}
logistic_predicted_class <- ifelse(logistic_high_probability >= 0.5, "High", "Low")
logistic_predicted_class <- factor(logistic_predicted_class, levels = c("Low", "High"))

logistic_valid <- is.finite(logistic_high_probability) & !is.na(test_glm_model$Sales_Class)
logistic_confusion <- table(
  Actual = factor(test_glm_model$Sales_Class[logistic_valid], levels = c("Low", "High")),
  Predicted = factor(logistic_predicted_class[logistic_valid], levels = c("Low", "High"))
)
write.csv(as.data.frame.matrix(logistic_confusion),
          file.path(output_dir, "logistic_regression_confusion_matrix.csv"), row.names = TRUE)
cat("\n--- Logistic Regression confusion matrix (test set) ---\n")
print(logistic_confusion)

log_tn <- as.numeric(logistic_confusion["Low", "Low"])
log_fp <- as.numeric(logistic_confusion["Low", "High"])
log_fn <- as.numeric(logistic_confusion["High", "Low"])
log_tp <- as.numeric(logistic_confusion["High", "High"])
log_accuracy <- safe_ratio(log_tp + log_tn, sum(logistic_confusion))
log_precision_high <- safe_ratio(log_tp, log_tp + log_fp)
log_recall_high <- safe_ratio(log_tp, log_tp + log_fn)
log_specificity_low <- safe_ratio(log_tn, log_tn + log_fp)
log_f1_high <- if (is.na(log_precision_high) || is.na(log_recall_high) ||
                   log_precision_high + log_recall_high == 0) {
  NA_real_
} else {
  2 * log_precision_high * log_recall_high / (log_precision_high + log_recall_high)
}
log_auc <- auc_from_ranks(logistic_high_probability[logistic_valid],
                          test_glm_model$Sales_Class[logistic_valid])

logistic_metrics <- data.frame(
  Model = "Logistic Regression (glm)",
  Train_Rows = nrow(train_glm_model),
  Test_Rows = sum(logistic_valid),
  Training_Median_Global_Sales_Threshold = sales_threshold,
  Classification_Threshold = 0.50,
  Accuracy = log_accuracy,
  Precision_High = log_precision_high,
  Recall_High = log_recall_high,
  Specificity_Low = log_specificity_low,
  F1_High = log_f1_high,
  AUC_High = log_auc,
  stringsAsFactors = FALSE
)

model_comparison_metrics <- rbind(model_metrics, logistic_metrics)
write.csv(model_comparison_metrics, file.path(output_dir, "model_comparison_metrics.csv"), row.names = FALSE)
# Keep the existing model_metrics.csv useful as the all-model comparison too.
write.csv(model_comparison_metrics, file.path(output_dir, "model_metrics.csv"), row.names = FALSE)
saveRDS(sales_logistic, file.path(output_dir, "video_game_sales_logistic_regression.rds"))

cat("\n--- Model performance comparison (held-out test set) ---\n")
print(model_comparison_metrics[, c("Model", "Accuracy", "Precision_High", "Recall_High",
                                   "Specificity_Low", "F1_High", "AUC_High")], row.names = FALSE)

comparison_metric_names <- c("Accuracy", "Precision_High", "Recall_High",
                             "Specificity_Low", "F1_High", "AUC_High")
comparison_long <- do.call(rbind, lapply(seq_len(nrow(model_comparison_metrics)), function(i) {
  data.frame(
    Model = model_comparison_metrics$Model[i],
    Metric = comparison_metric_names,
    Score = as.numeric(unlist(model_comparison_metrics[i, comparison_metric_names], use.names = FALSE)),
    stringsAsFactors = FALSE
  )
}))
comparison_long <- comparison_long[is.finite(comparison_long$Score), , drop = FALSE]
p_model_comparison <- ggplot(comparison_long,
  aes(x = Metric, y = Score, fill = Model)) +
  geom_col(position = position_dodge(width = 0.8), width = 0.72) +
  coord_cartesian(ylim = c(0, 1)) +
  scale_y_continuous(breaks = seq(0, 1, by = 0.2)) +
  scale_fill_manual(values = c("Decision Tree (rpart)" = "#3478C8",
                               "Logistic Regression (glm)" = "#18A999")) +
  labs(title = "Decision Tree vs Logistic Regression",
       subtitle = "Held-out test set; High sales is the positive class",
       x = NULL, y = "Score (0–1)", fill = NULL) +
  base_theme + theme(axis.text.x = element_text(angle = 25, hjust = 1))
save_plot(p_model_comparison, "model_comparison_metrics.png", width = 10, height = 6)
print(p_model_comparison)

# Logistic regression coefficient table and a compact top-coefficients chart.
# Coefficients compare each factor level with its reference level.
coef_values <- stats::coef(sales_logistic)
coef_df <- data.frame(
  Feature = names(coef_values),
  Coefficient = as.numeric(coef_values),
  stringsAsFactors = FALSE
)
coef_df <- coef_df[is.finite(coef_df$Coefficient) & coef_df$Feature != "(Intercept)", , drop = FALSE]
coef_df$Odds_Ratio <- exp(pmax(pmin(coef_df$Coefficient, 30), -30))
coef_df <- coef_df[order(abs(coef_df$Coefficient), decreasing = TRUE), , drop = FALSE]
write.csv(coef_df, file.path(output_dir, "logistic_regression_coefficients.csv"), row.names = FALSE)
if (nrow(coef_df) > 0) {
  coef_plot_df <- head(coef_df, 15)
  coef_plot_df$Direction <- ifelse(coef_plot_df$Coefficient >= 0, "Positive for High", "Negative for High")
  p_logistic_coefficients <- ggplot(coef_plot_df,
    aes(x = reorder(Feature, Coefficient), y = Coefficient, fill = Direction)) +
    geom_col() + coord_flip() +
    scale_fill_manual(values = c("Positive for High" = "#18A999", "Negative for High" = "#D95F59")) +
    labs(title = "Largest Logistic Regression Coefficients",
         subtitle = "Coefficients are relative to reference factor levels; associations are not causal",
         x = NULL, y = "Log-odds coefficient", fill = NULL) + base_theme
  save_plot(p_logistic_coefficients, "logistic_regression_coefficients.png", width = 9, height = 7)
  print(p_logistic_coefficients)
}

cat("Logistic Regression model saved and comparison chart written to the output folder.\n")


## 9. Model diagnostics and feature importance

Displays and saves the Decision Tree's test-set confusion matrix, variable importance, fitted tree, High/Low class balance, ROC curve, and metrics chart. The previous section adds the Logistic Regression confusion matrix, coefficient chart, and a side-by-side comparison of both classifiers.


In [ ]:
# 9. Model diagnostics and feature-importance charts (inline + PNG)
# ---------------------------------------------------------------------
confusion_df <- as.data.frame(confusion)
names(confusion_df) <- c("Actual", "Predicted", "Count")
p_cm <- ggplot(confusion_df, aes(x = Predicted, y = Actual, fill = Count)) +
  geom_tile(color = "white", linewidth = 1) +
  geom_text(aes(label = Count), size = 6, fontface = "bold") +
  scale_fill_gradient(low = "#DCEBFA", high = "#2166AC") +
  labs(title = "Confusion Matrix — Test Set", x = "Predicted class", y = "Actual class") + base_theme
save_plot(p_cm, "confusion_matrix.png", width = 7, height = 5.5)
print(p_cm)

variable_importance <- sales_tree$variable.importance
if (is.null(variable_importance) || length(variable_importance) == 0) {
  importance_df <- data.frame(Feature = character(0), Importance = numeric(0), Percent_Importance = numeric(0))
  message("The tree has no nonzero variable-importance values to plot.")
} else {
  importance_df <- data.frame(Feature = names(variable_importance),
                              Importance = as.numeric(variable_importance),
                              stringsAsFactors = FALSE)
  importance_df <- importance_df[order(importance_df$Importance, decreasing = TRUE), , drop = FALSE]
  importance_df$Percent_Importance <- 100 * importance_df$Importance / sum(importance_df$Importance)
  p_importance <- ggplot(importance_df, aes(x = reorder(Feature, Importance), y = Importance)) +
    geom_col(fill = "#18A999") + coord_flip() +
    labs(title = "Decision Tree Variable Importance", x = "Feature", y = "Importance") + base_theme
  save_plot(p_importance, "decision_tree_feature_importance.png", width = 8, height = 5.5)
  print(p_importance)
}
write.csv(importance_df, file.path(output_dir, "decision_tree_variable_importance.csv"), row.names = FALSE)

# Save and display the tree plot.
png(file.path(plot_dir, "decision_tree.png"), width = 1800, height = 1100, res = 150, bg = "white")
tryCatch(
  rpart.plot::rpart.plot(sales_tree, type = 3, extra = 104, fallen.leaves = TRUE,
                         main = "High / Low Global Sales Classification — Decision Tree"),
  error = function(e) {
    plot.new()
    text(0.5, 0.5, paste("Decision tree plot error:", conditionMessage(e)))
  },
  finally = grDevices::dev.off()
)
rpart.plot::rpart.plot(sales_tree, type = 3, extra = 104, fallen.leaves = TRUE,
                       main = "High / Low Global Sales Classification — Decision Tree")

saveRDS(sales_tree, file.path(output_dir, "video_game_sales_decision_tree.rds"))
writeLines(as.character(sales_threshold), file.path(output_dir, "training_median_sales_threshold.txt"))

# Target class balance
p_target <- ggplot(data.frame(Sales_Class = train_model$Sales_Class), aes(x = Sales_Class, fill = Sales_Class)) +
  geom_bar(show.legend = FALSE) +
  scale_fill_manual(values = c("Low" = "#3478C8", "High" = "#E58A2B")) +
  labs(title = "High / Low Sales Class Balance (Training Set)", x = "Sales class", y = "Number of games") + base_theme
save_plot(p_target, "sales_class_distribution.png", width = 7, height = 5)
print(p_target)

# ROC curve computed from held-out test probabilities
roc_cutoffs <- seq(0, 1, length.out = 101)
actual_high <- as.character(test_model$Sales_Class) == "High"
roc_df <- do.call(rbind, lapply(roc_cutoffs, function(thr) {
  pred_high <- high_probability >= thr
  positives <- sum(actual_high, na.rm = TRUE)
  negatives <- sum(!actual_high, na.rm = TRUE)
  data.frame(
    Threshold = thr,
    TPR = if (positives > 0) sum(pred_high & actual_high, na.rm = TRUE) / positives else NA_real_,
    FPR = if (negatives > 0) sum(pred_high & !actual_high, na.rm = TRUE) / negatives else NA_real_
  )
}))
roc_df <- roc_df[is.finite(roc_df$TPR) & is.finite(roc_df$FPR), , drop = FALSE]
if (nrow(roc_df) > 1) {
  p_roc <- ggplot(roc_df, aes(x = FPR, y = TPR)) +
    geom_line(color = "#3478C8", linewidth = 1.1) +
    geom_abline(intercept = 0, slope = 1, linetype = "dashed", color = "grey55") +
    coord_equal(xlim = c(0, 1), ylim = c(0, 1)) +
    labs(title = paste0("ROC Curve — Test Set (AUC = ", sprintf("%.3f", auc), ")"),
         x = "False Positive Rate", y = "True Positive Rate") + base_theme
  save_plot(p_roc, "roc_curve.png", width = 7, height = 6)
  print(p_roc)
} else {
  message("ROC chart skipped because the test set does not contain both classes.")
}

# Side-by-side performance metrics
metric_plot_df <- data.frame(
  Metric = c("Accuracy", "Precision (High)", "Recall (High)", "Specificity (Low)", "F1 (High)", "AUC (High)"),
  Value = as.numeric(c(accuracy, precision_high, recall_high, specificity_low, f1_high, auc)),
  stringsAsFactors = FALSE
)
metric_plot_df <- metric_plot_df[is.finite(metric_plot_df$Value), , drop = FALSE]
if (nrow(metric_plot_df) > 0) {
  p_metrics <- ggplot(metric_plot_df, aes(x = reorder(Metric, Value), y = Value)) +
    geom_col(fill = "#7B61A8") + coord_flip() +
    scale_y_continuous(limits = c(0, 1), breaks = seq(0, 1, 0.2)) +
    geom_text(aes(label = sprintf("%.3f", Value)), hjust = -0.15, size = 3.5) +
    expand_limits(y = 1.12) +
    labs(title = "Decision Tree Test-Set Metrics", x = NULL, y = "Score (0–1)") + base_theme
  save_plot(p_metrics, "model_performance_metrics.png", width = 8, height = 5.5)
  print(p_metrics)
}

cat("\nModel diagnostic charts saved to: ", normalizePath(plot_dir, mustWork = FALSE), "\n", sep = "")


## 10. LIME explainability for Logistic Regression

LIME (Local Interpretable Model-agnostic Explanations) explains individual predictions by approximating the classifier locally. This section explains four held-out examples: two with the highest predicted probability of High sales and two with the lowest. Blue/green or red bars indicate features that support or oppose the selected class according to the local explanation; they are not causal effects.


In [ ]:
# 10. LIME local explanations for selected test-set games
# ---------------------------------------------------------------------
# The R lime package natively supports caret models; this notebook uses glm.
# We register a small adapter so lime receives the class probabilities in
# its required format without changing the fitted Logistic Regression model.

predict_model.video_game_sales_glm_lime <- function(x, newdata, type, ...) {
  prob_high <- as.numeric(stats::predict(x$fitted_model, newdata = newdata, type = "response"))
  if (any(!is.finite(prob_high))) {
    stop("LIME generated non-finite predictions. Check factor levels or the fitted logistic model.")
  }
  prob_high <- pmin(pmax(prob_high, 0), 1)
  if (identical(type, "prob")) {
    return(data.frame(Low = 1 - prob_high, High = prob_high, check.names = FALSE))
  }
  if (identical(type, "raw")) {
    return(data.frame(Response = ifelse(prob_high >= 0.5, "High", "Low"),
                      stringsAsFactors = FALSE))
  }
  stop("Unsupported lime prediction type: ", type)
}

model_type.video_game_sales_glm_lime <- function(x, ...) "classification"

# Register custom adapter methods with the lime S3 generics.
base::registerS3method(
  "predict_model", "video_game_sales_glm_lime",
  predict_model.video_game_sales_glm_lime, envir = asNamespace("lime")
)
base::registerS3method(
  "model_type", "video_game_sales_glm_lime",
  model_type.video_game_sales_glm_lime, envir = asNamespace("lime")
)

lime_model_adapter <- structure(
  list(fitted_model = sales_logistic),
  class = "video_game_sales_glm_lime"
)

lime_predictor_columns <- c("Genre", "Platform", "Publisher", "Year")
lime_train_x <- as.data.frame(train_glm_model[, lime_predictor_columns, drop = FALSE])
lime_test_x <- as.data.frame(test_glm_model[, lime_predictor_columns, drop = FALSE])

lime_explainer <- lime::lime(
  x = lime_train_x,
  model = lime_model_adapter,
  bin_continuous = TRUE,
  n_bins = 4
)

# Pick two high-probability and two low-probability test cases.
valid_lime_rows <- which(is.finite(logistic_high_probability))
if (length(valid_lime_rows) < 2) {
  stop("Not enough valid test predictions to create LIME explanations.")
}
ordered_lime_rows <- valid_lime_rows[order(logistic_high_probability[valid_lime_rows], decreasing = TRUE)]
lime_indices <- unique(c(head(ordered_lime_rows, 2), tail(ordered_lime_rows, 2)))
lime_cases <- lime_test_x[lime_indices, , drop = FALSE]
rownames(lime_cases) <- paste0("TestCase_", seq_len(nrow(lime_cases)))

set.seed(42)
lime_explanations <- lime::explain(
  x = lime_cases,
  explainer = lime_explainer,
  labels = "High",
  n_features = 4,
  n_permutations = 500,
  feature_select = "auto"
)

cat("\n--- LIME explanations: selected test-set games ---\n")
print(lime_explanations[, intersect(
  c("case", "label", "label_prob", "model_prediction", "model_r2", "feature", "feature_value", "feature_weight", "feature_desc"),
  names(lime_explanations)
), drop = FALSE])

# Export only scalar columns so that the CSV is easy to open in Excel.
lime_export_columns <- intersect(
  c("case", "label", "label_prob", "model_prediction", "model_r2", "feature", "feature_value", "feature_weight", "feature_desc"),
  names(lime_explanations)
)
lime_export <- lime_explanations[, lime_export_columns, drop = FALSE]
write.csv(lime_export, file.path(output_dir, "lime_logistic_regression_explanations.csv"), row.names = FALSE)

p_lime <- lime::plot_features(lime_explanations, ncol = 2) +
  ggplot2::labs(
    title = "LIME: Why did Logistic Regression predict High sales?",
    subtitle = "Local feature contributions for selected test-set records",
    x = "Local explanation weight", y = NULL
  ) +
  ggplot2::theme_minimal(base_size = 11) +
  ggplot2::theme(
    plot.title = ggplot2::element_text(face = "bold"),
    strip.text = ggplot2::element_text(face = "bold", size = 9),
    panel.spacing = grid::unit(1.2, "lines"),
    legend.position = "bottom"
  )

ggplot2::ggsave(
  filename = file.path(plot_dir, "lime_logistic_regression_explanations.png"),
  plot = p_lime, width = 12, height = 8, units = "in", dpi = 180, bg = "white"
)
print(p_lime)
cat("LIME explanations saved to: ", file.path(output_dir, "lime_logistic_regression_explanations.csv"), "\n", sep = "")


## 11. Generate findings summary

Builds a text report from the actual dataset results, including the top genre, platform, sales year, regional total, and test metrics.


In [ ]:
# 11. Automatically generated findings summary
# ---------------------------------------------------------------------
top_genre_row <- if (nrow(genre_summary) > 0) genre_summary[1, , drop = FALSE] else NULL
top_platform_row <- if (nrow(platform_summary) > 0) platform_summary[1, , drop = FALSE] else NULL
top_year_row <- if (nrow(yearly_summary) > 0) yearly_summary[which.max(yearly_summary$Total_Global_Sales), , drop = FALSE] else NULL
top_region_row <- if (nrow(regional_summary) > 0) regional_summary[1, , drop = FALSE] else NULL

findings <- c(
  "VIDEO GAME SALES CASE STUDY — RUN SUMMARY",
  "==========================================",
  paste0("Input dataset: ", normalizePath(dataset_path, mustWork = FALSE)),
  paste0("Rows imported: ", rows_imported),
  paste0("Rows retained after exact-duplicate and invalid Global_Sales handling: ", nrow(clean)),
  paste0("Exact duplicate rows removed: ", rows_removed_duplicates),
  paste0("Rows removed for missing/invalid Global_Sales: ", rows_removed_invalid_global),
  paste0("Global_Sales high/low threshold (training median): ", signif(sales_threshold, 6)),
  "",
  "RESULTS CALCULATED FROM THIS DATA FILE",
  if (!is.null(top_genre_row)) paste0("Genre with highest total Global_Sales: ", top_genre_row$Genre,
                                     " (", signif(top_genre_row$Total_Global_Sales, 6), ")") else "Genre summary unavailable.",
  if (!is.null(top_platform_row)) paste0("Platform with highest total Global_Sales: ", top_platform_row$Platform,
                                         " (", signif(top_platform_row$Total_Global_Sales, 6), ")") else "Platform summary unavailable.",
  if (!is.null(top_year_row)) paste0("Year with highest recorded total Global_Sales: ", top_year_row$Year,
                                     " (", signif(top_year_row$Total_Global_Sales, 6), ")") else "Year summary unavailable.",
  if (!is.null(top_region_row)) paste0("Region with highest total sales among provided regional columns: ", top_region_row$Region,
                                       " (", signif(top_region_row$Total_Regional_Sales, 6), ")") else "Regional sales summary unavailable.",
  paste0("Decision tree test accuracy: ", ifelse(is.na(accuracy), "NA", sprintf("%.3f", accuracy))),
  paste0("Decision tree test precision (High): ", ifelse(is.na(precision_high), "NA", sprintf("%.3f", precision_high))),
  paste0("Decision tree test recall (High): ", ifelse(is.na(recall_high), "NA", sprintf("%.3f", recall_high))),
  paste0("Decision tree test F1 (High): ", ifelse(is.na(f1_high), "NA", sprintf("%.3f", f1_high))),
  paste0("Decision tree test AUC (High): ", ifelse(is.na(auc), "NA", sprintf("%.3f", auc))),
  paste0("Logistic regression test accuracy: ", ifelse(is.na(log_accuracy), "NA", sprintf("%.3f", log_accuracy))),
  paste0("Logistic regression test precision (High): ", ifelse(is.na(log_precision_high), "NA", sprintf("%.3f", log_precision_high))),
  paste0("Logistic regression test recall (High): ", ifelse(is.na(log_recall_high), "NA", sprintf("%.3f", log_recall_high))),
  paste0("Logistic regression test F1 (High): ", ifelse(is.na(log_f1_high), "NA", sprintf("%.3f", log_f1_high))),
  paste0("Logistic regression test AUC (High): ", ifelse(is.na(log_auc), "NA", sprintf("%.3f", log_auc))),
  "",
  "INTERPRETATION NOTES",
  "- High/Low sales threshold is calculated from the training-set median.",
  "- Global_Sales is used to construct the target and is excluded from model predictors.",
  "- Rank and regional sales columns are excluded from the model to reduce target leakage.",
  "- Both models are evaluated on the same held-out records; differences reflect model choice and the documented categorical-level grouping.",
  "- LIME provides local model approximations for selected records; explanations are not causal effects.",
  "- These results describe patterns in the selected historical dataset and do not guarantee future success.",
  "- Confirm sales units from the Kaggle dataset documentation before presenting totals."
)
writeLines(findings, file.path(output_dir, "case_study_findings.txt"))

cat("\n--- Findings generated from the selected dataset ---\n")
cat(paste(findings, collapse = "\n"), "\n")


## 12. Final output inventory

Prints the location of the output folder and lists the key artifacts created by the analysis.


In [ ]:
# 12. Final output inventory
# ---------------------------------------------------------------------
cat("\n============================================================\n")
cat("CASE STUDY COMPLETED\n")
cat("============================================================\n")
cat("All outputs saved in: ", normalizePath(output_dir, mustWork = FALSE), "\n", sep = "")
cat("Key files:\n")
cat(" - cleaned_video_game_sales.csv\n")
cat(" - missing_values_by_column.csv\n")
cat(" - data_quality_summary.csv\n")
cat(" - genre_sales_summary.csv\n")
cat(" - platform_sales_summary.csv\n")
cat(" - publisher_sales_summary.csv\n")
cat(" - yearly_sales_summary.csv\n")
cat(" - regional_sales_summary.csv\n")
cat(" - top_10_games_by_global_sales.csv\n")
cat(" - model_metrics.csv (tree and logistic regression comparison)\n")
cat(" - model_comparison_metrics.csv\n")
cat(" - logistic_regression_confusion_matrix.csv\n")
cat(" - logistic_regression_coefficients.csv\n")
cat(" - lime_logistic_regression_explanations.csv\n")
cat(" - confusion_matrix.csv\n")
cat(" - decision_tree_variable_importance.csv\n")
cat(" - video_game_sales_decision_tree.rds\n")
cat(" - video_game_sales_logistic_regression.rds\n")
cat(" - case_study_findings.txt\n")
cat("Plots saved in the plots/ subfolder (PNG files), including model comparison and LIME explanations.\n")
cat("\nRemember: the project brief expects results to be interpreted from the actual CSV, not assumed in advance.\n")


## 13. Review and open generated outputs

The EDA and model charts are displayed inline as their cells run. Outputs include the original Decision Tree baseline, a Logistic Regression comparison, a confusion matrix and coefficient table for Logistic Regression, and LIME plots explaining selected individual test predictions. All CSV summaries, fitted `.rds` models, the findings report, and PNG charts are written inside `video_game_sales_outputs/`.


In [ ]:
# Confirm the output inventory after all analysis and charts have run.
if (!exists("output_dir") || !dir.exists(output_dir)) {
  stop("The output folder does not exist yet. Run the analysis cells above first.")
}

cat("Output folder:\n", normalizePath(output_dir, mustWork = FALSE), "\n\n", sep = "")
files_created <- list.files(output_dir, recursive = TRUE)
cat("Files created (", length(files_created), "):\n", sep = "")
print(files_created)

plot_files <- list.files(file.path(output_dir, "plots"), pattern = "\\.png$",
                         full.names = TRUE, ignore.case = TRUE)
cat("\nNumber of PNG charts saved: ", length(plot_files), "\n", sep = "")
if (length(plot_files) > 0) {
  cat("Charts are displayed inline in the EDA and model-diagnostics cells.\n")
  cat("Open the Files panel and navigate to: ", normalizePath(file.path(output_dir, "plots"), mustWork = FALSE), "\n", sep = "")
} else {
  message("No plot files were found. Check that the EDA and model-diagnostics cells completed successfully.")
}
